# Purchase Intent - Optimized

## Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import ExtraTreesRegressor, HistGradientBoostingRegressor, HistGradientBoostingClassifier
from sklearn.multioutput import MultiOutputRegressor
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import r2_score, accuracy_score, classification_report, f1_score
from imblearn.over_sampling import SMOTE
import joblib

df = pd.read_csv("../../data/processed/cleaned_users.csv")

input_features = ["Age", "Monthly_Income", "Annual_Income", "Family_Size",
                   "Plot_Budget", "Preferred_Plot_Size_SqFt", "Distance_to_City_Center_km",
                   "Gender", "Occupation", "City", "Current_Housing_Status",
                   "Preferred_Location", "Purpose", "Loan_Required",
                   "Expected_Purchase_Timeline", "Lead_Source",
                   "Previous_Enquiry", "Site_Visit", "Negotiation_Done"]
cat_cols = ["Gender", "Occupation", "City", "Current_Housing_Status", "Preferred_Location",
            "Purpose", "Loan_Required", "Expected_Purchase_Timeline", "Lead_Source",
            "Previous_Enquiry", "Site_Visit", "Negotiation_Done"]


## Part 1 - Regression Baseline

In [ ]:
target_cols = ["Purchase_Probability", "Purchase_Value"]
X = pd.get_dummies(df[input_features], columns=cat_cols, drop_first=True)
y = df[target_cols].copy()
y.loc[df["Purchased"] == "No", "Purchase_Value"] = 0

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = MinMaxScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

prev_reg = joblib.load("../../models/purchase_multi_output_model.pkl")
prev_reg_pred = prev_reg.predict(X_test_s)
prev_reg_r2 = r2_score(y_test, prev_reg_pred, multioutput="uniform_average")
print(f"Original Best (purchase_intent_model.ipynb): avg R2={prev_reg_r2:.3f}")


## New Techniques - Regression

In [ ]:
et_model = ExtraTreesRegressor(n_estimators=200, max_depth=15, random_state=42)
et_model.fit(X_train_s, y_train)
et_r2 = r2_score(y_test, et_model.predict(X_test_s), multioutput="uniform_average")
print(f"Extra Trees: avg R2={et_r2:.3f}")

hgb_model = MultiOutputRegressor(HistGradientBoostingRegressor(max_iter=150, random_state=42))
hgb_model.fit(X_train_s, y_train)
hgb_r2 = r2_score(y_test, hgb_model.predict(X_test_s), multioutput="uniform_average")
print(f"HistGradientBoosting: avg R2={hgb_r2:.3f}")

reg_candidates = {"Original": (prev_reg, prev_reg_r2), "Extra Trees": (et_model, et_r2), "HistGradientBoosting": (hgb_model, hgb_r2)}
best_reg_name = max(reg_candidates, key=lambda k: reg_candidates[k][1])
best_reg_model, best_reg_r2 = reg_candidates[best_reg_name]
print(f"Best: {best_reg_name} (R2={best_reg_r2:.3f})")

if best_reg_name != "Original":
    joblib.dump(best_reg_model, "../../models/purchase_multi_output_model_v2.pkl")
    print(f"Saved as *_v2.pkl - improved R2 {prev_reg_r2:.3f} -> {best_reg_r2:.3f}")


## Part 2 - Classification Baseline

In [ ]:
yc = df["Purchase_Intent"]
Xc = pd.get_dummies(df[input_features], columns=cat_cols, drop_first=True)
Xc_train, Xc_test, yc_train, yc_test = train_test_split(Xc, yc, test_size=0.2, random_state=42, stratify=yc)
clf_scaler = MinMaxScaler()
Xc_train_s = clf_scaler.fit_transform(Xc_train)
Xc_test_s = clf_scaler.transform(Xc_test)

prev_clf = joblib.load("../../models/purchase_intent_classifier.pkl")
prev_pred = prev_clf.predict(Xc_test_s)
prev_acc = accuracy_score(yc_test, prev_pred)
prev_f1 = f1_score(yc_test, prev_pred, average="macro")
print(f"Original Best: accuracy={prev_acc:.3f}  macro-F1={prev_f1:.3f}")
print(classification_report(yc_test, prev_pred))


## New Technique - SMOTE + Calibration

In [ ]:
X_res, y_res = SMOTE(random_state=42).fit_resample(Xc_train_s, yc_train)
pd.Series(y_res).value_counts()


In [ ]:
base_clf = HistGradientBoostingClassifier(max_iter=150, random_state=42)
smote_clf = CalibratedClassifierCV(base_clf, cv=3)
smote_clf.fit(X_res, y_res)
smote_pred = smote_clf.predict(Xc_test_s)
smote_acc = accuracy_score(yc_test, smote_pred)
smote_f1 = f1_score(yc_test, smote_pred, average="macro")
print(f"SMOTE + Calibrated HistGB: accuracy={smote_acc:.3f}  macro-F1={smote_f1:.3f}")
print(classification_report(yc_test, smote_pred))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11,4))
sns.heatmap(pd.crosstab(yc_test, prev_pred, normalize="index"), annot=True, fmt=".2f", cmap="Blues", ax=axes[0])
axes[0].set_title(f"Original (macro-F1={prev_f1:.3f})")
sns.heatmap(pd.crosstab(yc_test, smote_pred, normalize="index"), annot=True, fmt=".2f", cmap="Blues", ax=axes[1])
axes[1].set_title(f"SMOTE + Calibrated (macro-F1={smote_f1:.3f})")
plt.tight_layout(); plt.show()


## Save

In [ ]:
if smote_f1 > prev_f1:
    joblib.dump(smote_clf, "../models/purchase_intent_classifier_v2.pkl")
    joblib.dump(clf_scaler, "../models/purchase_intent_clf_scaler_v2.pkl")
    joblib.dump(list(Xc.columns), "../models/purchase_intent_clf_columns_v2.pkl")
    print(f"Saved as *_v2.pkl - improved macro-F1 {prev_f1:.3f} -> {smote_f1:.3f}")
else:
    print(f"Original model kept - new macro-F1 {smote_f1:.3f} did not beat {prev_f1:.3f}")
